In [1]:
import os, sys, json, time, shutil
from pathlib import Path
PROJECT = Path(r"C:\Users\hp\Documents\teliport_prototype")
os.chdir(PROJECT / "notebooks")
sys.path.insert(0, str(PROJECT / "src"))
%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd
from cmapss_data import load_all, SUBSETS

DATA_DIR = PROJECT / "data" / "raw" / "CMAPSS"
MODELS = PROJECT / "models"
REPORTS = PROJECT / "reports"
WO_DIR = PROJECT / "work_orders"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

# Step 8A replaced the models, so old MLOps deployment labels are stale -> remove them
for f in MODELS.glob("deployed_*.json"):
    f.unlink()
data = load_all(DATA_DIR)
print("Working dir:", Path.cwd())

Working dir: C:\Users\hp\Documents\teliport_prototype\notebooks


In [2]:
%%writefile ../src/inference.py
"""Fleet scoring: raw sensor history -> one snapshot per engine (RUL range + health + root cause)."""
from __future__ import annotations

import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from features import last_cycle
from modeling import predict


def load_models(fd, models_dir):
    models_dir = Path(models_dir)
    unc_path = models_dir / f"uncertainty_{fd}.joblib"
    return (joblib.load(models_dir / f"preprocessor_{fd}.joblib"),
            joblib.load(models_dir / f"lgbm_{fd}.joblib"),
            joblib.load(models_dir / f"health_monitor_{fd}.joblib"),
            joblib.load(unc_path) if unc_path.exists() else None)


def deployed_version(fd, models_dir):
    """Registry version written by MLOps deploy(); 'local' if the models were trained in a notebook."""
    path = Path(models_dir) / f"deployed_{fd}.json"
    return f"registry v{json.loads(path.read_text())['version']}" if path.exists() else "local"


def score_fleet(raw_df, fd, models_dir, model_version=None):
    """raw_df = sensor history (unit, cycle, settings, sensors) for any number of engines."""
    prep, rul_model, hm, unc = load_models(fd, models_dir)
    model_version = model_version or deployed_version(fd, models_dir)
    feats = prep.transform(raw_df)
    flags = hm.transform(feats)
    last = last_cycle(feats)
    last_flags = flags.groupby("unit").tail(1).set_index("unit")
    snap = pd.DataFrame({
        "subset": fd,
        "unit": last["unit"].to_numpy(),
        "cycles_seen": last["cycle"].astype(int).to_numpy(),
        "pred_RUL": np.round(predict(rul_model, last, prep.feature_names_), 1),
    })
    if unc is not None:                                   # 90% prediction interval
        lower, upper = unc.interval(snap["pred_RUL"].to_numpy())
        snap["rul_lower"], snap["rul_upper"] = np.round(lower, 1), np.round(upper, 1)
    else:
        snap["rul_lower"] = snap["rul_upper"] = snap["pred_RUL"]
    snap["anomaly_score"] = np.round(last_flags.loc[snap["unit"], "anomaly_score"].to_numpy(), 2)
    snap["alarm"] = last_flags.loc[snap["unit"], "alarm"].to_numpy().astype(bool)
    snap["top_sensors"] = [hm.top_contributors(row) for _, row in last.iterrows()]
    snap["model_version"] = model_version
    return snap

Overwriting ../src/inference.py


In [3]:
%%writefile ../src/triage_agent.py
"""Rule-based Triage Agent (no LLM).

Reads model outputs for each engine, decides severity with transparent threshold rules,
flags exceptions for a human, and writes maintenance work orders (JSON + CSV log).
v2: uses the 90% lower bound of the RUL prediction for the safety-critical rule.
"""
from __future__ import annotations

import json
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

SEVERITY_ORDER = {"LOW": 0, "MEDIUM": 1, "HIGH": 2, "CRITICAL": 3}

SENSOR_COMPONENT = {
    "s2": "Low-pressure compressor (LPC)", "s3": "High-pressure compressor (HPC)",
    "s4": "Low-pressure turbine (LPT)", "s7": "High-pressure compressor (HPC)",
    "s8": "Fan", "s9": "Engine core (HPC/HPT spool)", "s11": "High-pressure compressor (HPC)",
    "s12": "High-pressure compressor (HPC)", "s13": "Fan", "s14": "Engine core (HPC/HPT spool)",
    "s15": "Fan / bypass duct", "s17": "Bleed air system",
    "s20": "High-pressure turbine (HPT) cooling", "s21": "Low-pressure turbine (LPT) cooling",
}


@dataclass
class TriageConfig:
    critical_rul: float = 20      # <= 20 cycles left  -> CRITICAL
    high_rul: float = 40          # <= 40              -> HIGH
    medium_rul: float = 75        # <= 75              -> MEDIUM, else LOW (monitor only)
    due_fraction: float = 0.5     # schedule work at half the decision RUL (2x safety factor)
    disagreement_rul: float = 100 # alarm while RUL > 100 -> models disagree -> human review
    min_cycles: int = 20          # shorter history -> low confidence -> human review
    min_severity_for_wo: str = "MEDIUM"
    conservative_mode: str = "critical" # "critical": 90% lower bound used for the CRITICAL rule only
                                        # "all": lower bound for every rule, "none": point estimate only


@dataclass
class Decision:
    subset: str
    unit: int
    cycles_seen: int
    pred_RUL: float
    rul_lower: float
    rul_upper: float
    decision_RUL: float
    anomaly_score: float
    alarm: bool
    severity: str
    suspected_component: str
    top_sensors: list
    needs_human_review: bool
    review_reasons: list = field(default_factory=list)
    rules_fired: list = field(default_factory=list)
    model_version: str = "unknown"


class TriageAgent:
    def __init__(self, config: TriageConfig | None = None, out_dir=None):
        self.cfg = config or TriageConfig()
        self.out_dir = Path(out_dir) if out_dir else None

    # ---------- 1. decide ----------
    def assess(self, snap: dict) -> Decision:
        c, alarm = self.cfg, bool(snap["alarm"])
        point = float(snap["pred_RUL"])
        lower = float(snap.get("rul_lower", point))
        upper = float(snap.get("rul_upper", point))
        rules, reasons = [], []

        def band(r):
            return ("CRITICAL" if r <= c.critical_rul else "HIGH" if r <= c.high_rul
                    else "MEDIUM" if r <= c.medium_rul else "LOW")

        if c.conservative_mode == "all" or (c.conservative_mode == "critical" and lower <= c.critical_rul):
            rul, name = lower, "conservative RUL (90% lower bound)"
        else:
            rul, name = point, "pred_RUL"
        sev = band(rul)
        limit = {"CRITICAL": c.critical_rul, "HIGH": c.high_rul, "MEDIUM": c.medium_rul}.get(sev)
        rule_id = {"CRITICAL": "R1", "HIGH": "R2", "MEDIUM": "R3", "LOW": "R4"}[sev]
        rules.append(f"{rule_id} {name} {rul:.0f} <= {limit:.0f}" if limit is not None
                     else f"R4 {name} {rul:.0f} > {c.medium_rul:.0f} (healthy)")
        if name.startswith("conservative") and band(point) != sev:
            rules.append(f"U1 point estimate {point:.0f} alone would give {band(point)}; "
                         "uncertainty pushes it to " + sev)

        if alarm and sev == "LOW":
            sev = "MEDIUM"; rules.append("E1 anomaly alarm active -> escalate LOW to MEDIUM (inspect)")

        if alarm and point > c.disagreement_rul:
            reasons.append("Health monitor alarm but RUL model sees > "
                           f"{c.disagreement_rul:.0f} cycles: possible sensor fault or unseen failure mode")
        if rul <= c.high_rul and not alarm:
            reasons.append("RUL model predicts near-failure but no anomaly alarm: verify sensors/data")
        if int(snap["cycles_seen"]) < c.min_cycles:
            reasons.append(f"Only {int(snap['cycles_seen'])} cycles of history: low-confidence prediction")
        rules += [f"X{i+1} human review: {r}" for i, r in enumerate(reasons)]

        top = list(snap.get("top_sensors") or [])
        weights = {}
        for s, _, pct in top:
            comp = SENSOR_COMPONENT.get(s, "Unknown")
            weights[comp] = weights.get(comp, 0) + pct
        component = max(weights, key=weights.get) if weights else "Unknown"

        return Decision(subset=str(snap["subset"]), unit=int(snap["unit"]),
                        cycles_seen=int(snap["cycles_seen"]), pred_RUL=round(point, 1),
                        rul_lower=round(lower, 1), rul_upper=round(upper, 1), decision_RUL=round(rul, 1),
                        anomaly_score=round(float(snap["anomaly_score"]), 2), alarm=alarm,
                        severity=sev, suspected_component=component,
                        top_sensors=[{"sensor": s, "name": n, "share_%": p} for s, n, p in top],
                        needs_human_review=bool(reasons), review_reasons=reasons, rules_fired=rules,
                        model_version=str(snap.get("model_version", "unknown")))

    # ---------- 2. act ----------
    def recommended_action(self, d: Decision, due: int) -> str:
        comp = d.suspected_component
        if d.severity == "CRITICAL":
            return f"Remove from service at next stop; inspect and repair/replace {comp} within {due} cycles."
        if d.severity == "HIGH":
            return f"Schedule maintenance on {comp} within {due} cycles; reserve parts and crew now."
        return f"Plan borescope inspection of {comp} within {due} cycles; increase monitoring frequency."

    def create_work_order(self, d: Decision, now: datetime | None = None) -> dict | None:
        if SEVERITY_ORDER[d.severity] < SEVERITY_ORDER[self.cfg.min_severity_for_wo]:
            return None
        now = now or datetime.now(timezone.utc)
        due = max(1, int(d.decision_RUL * self.cfg.due_fraction))
        wo = {
            "work_order_id": f"WO-{d.subset}-U{d.unit:03d}-C{d.cycles_seen:03d}",
            "created_at": now.isoformat(timespec="seconds"),
            "asset": f"{d.subset}/engine-{d.unit:03d}",
            "severity": d.severity,
            "status": "PENDING_HUMAN_REVIEW" if d.needs_human_review else "AUTO_APPROVED",
            "due_within_cycles": due,
            "recommended_action": self.recommended_action(d, due),
            **{k: v for k, v in asdict(d).items() if k not in ("subset", "unit")},
        }
        if self.out_dir:
            self.out_dir.mkdir(parents=True, exist_ok=True)
            (self.out_dir / f"{wo['work_order_id']}.json").write_text(json.dumps(wo, indent=2))
        return wo

    # ---------- 3. run over a fleet snapshot ----------
    def run(self, snapshots: pd.DataFrame) -> pd.DataFrame:
        rows, orders = [], []
        for snap in snapshots.to_dict("records"):
            d = self.assess(snap)
            wo = self.create_work_order(d)
            rows.append({"subset": d.subset, "unit": d.unit, "cycles_seen": d.cycles_seen,
                         "pred_RUL": d.pred_RUL, "rul_lower": d.rul_lower, "rul_upper": d.rul_upper,
                         "anomaly_score": d.anomaly_score, "alarm": d.alarm,
                         "severity": d.severity, "suspected_component": d.suspected_component,
                         "needs_human_review": d.needs_human_review,
                         "work_order_id": wo["work_order_id"] if wo else None,
                         "wo_status": wo["status"] if wo else "NO_ACTION",
                         "due_within_cycles": wo["due_within_cycles"] if wo else None})
            if wo:
                orders.append(wo)
        decisions = pd.DataFrame(rows)
        if self.out_dir and orders:
            log = pd.DataFrame([{k: (json.dumps(v) if isinstance(v, (list, dict)) else v)
                                 for k, v in o.items()} for o in orders])
            log_path = self.out_dir / "work_order_log.csv"
            if log_path.exists():                       # idempotent: never log the same order twice
                seen = set(pd.read_csv(log_path, usecols=["work_order_id"])["work_order_id"])
                log = log[~log["work_order_id"].isin(seen)]
            if len(log):
                log.to_csv(log_path, mode="a", header=not log_path.exists(), index=False)
        return decisions

Overwriting ../src/triage_agent.py


In [5]:
from triage_agent import TriageAgent, TriageConfig, SEVERITY_ORDER as SO

agent = TriageAgent()
base = dict(subset="FD001", unit=1, cycles_seen=150, anomaly_score=0.5, alarm=False,
            top_sensors=[("s11", "HPC static pressure (Ps30)", 60.0), ("s4", "LPT outlet temperature (T50)", 40.0)])
cases = [
    ("near failure + alarm",             dict(pred_RUL=8,   alarm=True,  anomaly_score=9.0), "CRITICAL", False),
    ("high risk + alarm",                dict(pred_RUL=35,  alarm=True,  anomaly_score=3.0), "HIGH",     False),
    ("medium risk + alarm",              dict(pred_RUL=60,  alarm=True,  anomaly_score=1.5), "MEDIUM",   False),
    ("healthy, no alarm",                dict(pred_RUL=118),                                  "LOW",      False),
    ("healthy RUL but alarm (90)",       dict(pred_RUL=90,  alarm=True,  anomaly_score=1.8), "MEDIUM",   False),
    ("models disagree (alarm, RUL 120)", dict(pred_RUL=120, alarm=True,  anomaly_score=4.0), "MEDIUM",   True),
    ("near failure but NO alarm",        dict(pred_RUL=15,  alarm=False),                     "CRITICAL", True),
    ("short history",                    dict(pred_RUL=50,  alarm=True,  cycles_seen=12),     "MEDIUM",   True),
    ("exact boundary RUL=20",            dict(pred_RUL=20,  alarm=True),                      "CRITICAL", False),
    ("exact boundary RUL=75",            dict(pred_RUL=75),                                   "MEDIUM",   False),
    ("NEW: point 26 but lower bound 17", dict(pred_RUL=26, rul_lower=17, rul_upper=35, alarm=True), "CRITICAL", False),
    ("NEW: point 26, lower bound 22",    dict(pred_RUL=26, rul_lower=22, rul_upper=35, alarm=True), "HIGH",     False),
    ("NEW: wide interval far from fail", dict(pred_RUL=60, rul_lower=45, rul_upper=75, alarm=True), "MEDIUM",   False),
]
rows = []
for name, over, exp_sev, exp_review in cases:
    d = agent.assess({**base, **over})
    rows.append({"case": name, "severity": d.severity, "expected": exp_sev, "decision_RUL": d.decision_RUL,
                 "review": d.needs_human_review,
                 "PASS": (d.severity == exp_sev) and (d.needs_human_review == exp_review)})
tests = pd.DataFrame(rows)
assert tests["PASS"].all(), "Some agent rule tests failed!"
print(f"All {len(tests)} agent rule tests passed.")
tests

All 13 agent rule tests passed.


,case,severity,expected,decision_RUL,review,PASS
0,near failure + alarm,CRITICAL,CRITICAL,8.0,False,True
1,high risk + alarm,HIGH,HIGH,35.0,False,True
2,medium risk + alarm,MEDIUM,MEDIUM,60.0,False,True
3,"healthy, no alarm",LOW,LOW,118.0,False,True
4,healthy RUL but alarm (90),MEDIUM,MEDIUM,90.0,False,True
5,"models disagree (alarm, RUL 120)",MEDIUM,MEDIUM,120.0,True,True
6,near failure but NO alarm,CRITICAL,CRITICAL,15.0,True,True
7,short history,MEDIUM,MEDIUM,50.0,True,True
8,exact boundary RUL=20,CRITICAL,CRITICAL,20.0,False,True
9,exact boundary RUL=75,MEDIUM,MEDIUM,75.0,False,True


In [6]:
from inference import score_fleet

snaps = pd.concat([score_fleet(data[fd][1], fd, MODELS) for fd in SUBSETS], ignore_index=True)
truth = pd.concat([data[fd][2].assign(subset=fd) for fd in SUBSETS])[["subset", "unit", "RUL"]]

def evaluate_agent(decisions, cfg):
    ev = decisions.merge(truth, on=["subset", "unit"])
    band = lambda r: ("CRITICAL" if r <= cfg.critical_rul else "HIGH" if r <= cfg.high_rul
                      else "MEDIUM" if r <= cfg.medium_rul else "LOW")
    ev["true_severity"] = ev["RUL"].apply(band)
    s, t = ev["severity"].map(SO), ev["true_severity"].map(SO)
    summary = {"exact_severity_match_%": round(100 * (s == t).mean(), 1),
               "within_one_level_%": round(100 * ((s - t).abs() <= 1).mean(), 1),
               "critical_missed (MEDIUM/LOW)": int(((ev.RUL <= cfg.critical_rul) & (s <= 1)).sum()),
               "critical_labelled_HIGH": int(((ev.RUL <= cfg.critical_rul) & (s == 2)).sum()),
               "under_triaged_%": round(100 * (s < t).mean(), 1),
               "over_triaged_%": round(100 * (s > t).mean(), 1),
               "work_orders": int(ev["work_order_id"].notna().sum()),
               "work_orders_failing_before_due": int((ev.work_order_id.notna() & (ev.RUL < ev.due_within_cycles)).sum()),
               "human_review_rate_%": round(100 * ev["needs_human_review"].mean(), 1)}
    return summary, ev

policies = {"v1: point estimate": "none", "v2: lower bound for CRITICAL": "critical",
            "lower bound everywhere": "all"}
results = {}
for name, mode in policies.items():
    cfg = TriageConfig(conservative_mode=mode)
    results[name], _ = evaluate_agent(TriageAgent(cfg).run(snaps), cfg)
policy_table = pd.DataFrame(results)
policy_table.to_csv(REPORTS / "agent_policy_comparison.csv")
policy_table

,v1: point estimate,v2: lower bound for CRITICAL,lower bound everywhere
exact_severity_match_%,88.3,85.4,72.8
within_one_level_%,99.9,99.9,99.9
critical_missed (MEDIUM/LOW),0.0,0.0,0.0
critical_labelled_HIGH,7.0,0.0,0.0
under_triaged_%,5.5,4.5,0.4
over_triaged_%,6.2,10.0,26.7
work_orders,331.0,331.0,435.0
work_orders_failing_before_due,0.0,0.0,0.0
human_review_rate_%,1.0,1.0,1.8


In [7]:
shutil.rmtree(WO_DIR, ignore_errors=True)                # fresh run
agent = TriageAgent(TriageConfig(conservative_mode="critical"), out_dir=WO_DIR)
decisions = agent.run(snaps)
decisions.to_csv(REPORTS / "triage_decisions_test.csv", index=False)

print("Work-order files written:", len(list(WO_DIR.glob("WO-*.json"))))
print(decisions["wo_status"].value_counts(), "\n")

up = decisions[(decisions["severity"] == "CRITICAL") & (decisions["pred_RUL"] > 20)]
print(f"{len(up)} engines were escalated to CRITICAL because of uncertainty. Example work order:")
wo = json.loads((WO_DIR / f"{up.iloc[0].work_order_id}.json").read_text())
print(json.dumps({k: wo[k] for k in ["work_order_id", "severity", "status", "pred_RUL", "rul_lower",
                                     "rul_upper", "decision_RUL", "due_within_cycles",
                                     "recommended_action", "rules_fired", "model_version"]}, indent=2))

Work-order files written: 331
wo_status
NO_ACTION               376
AUTO_APPROVED           326
PENDING_HUMAN_REVIEW      5
Name: count, dtype: int64 

34 engines were escalated to CRITICAL because of uncertainty. Example work order:
{
  "work_order_id": "WO-FD001-U024-C186",
  "severity": "CRITICAL",
  "status": "AUTO_APPROVED",
  "pred_RUL": 22.4,
  "rul_lower": 15.1,
  "rul_upper": 31.4,
  "decision_RUL": 15.1,
  "due_within_cycles": 7,
  "recommended_action": "Remove from service at next stop; inspect and repair/replace Engine core (HPC/HPT spool) within 7 cycles.",
  "rules_fired": [
    "R1 conservative RUL (90% lower bound) 15 <= 20",
    "U1 point estimate 22 alone would give HIGH; uncertainty pushes it to CRITICAL"
  ],
  "model_version": "local"
}


In [8]:
cfg = TriageConfig(conservative_mode="critical")
summary, ev = evaluate_agent(decisions, cfg)
pd.Series(summary).to_csv(REPORTS / "agent_evaluation_v2.csv")
for k, v in summary.items():
    print(f"{k:34s} {v}")

order = ["CRITICAL", "HIGH", "MEDIUM", "LOW"]
print("\nConfusion matrix (rows = true severity, cols = agent severity):")
pd.crosstab(ev["true_severity"], ev["severity"]).reindex(index=order, columns=order, fill_value=0)

exact_severity_match_%             85.4
within_one_level_%                 99.9
critical_missed (MEDIUM/LOW)       0
critical_labelled_HIGH             0
under_triaged_%                    4.5
over_triaged_%                     10.0
work_orders                        331
work_orders_failing_before_due     0
human_review_rate_%                1.0

Confusion matrix (rows = true severity, cols = agent severity):


severity,CRITICAL,HIGH,MEDIUM,LOW
true_severity,,,,
CRITICAL,123,0,0,0
HIGH,29,36,7,0
MEDIUM,0,6,94,25
LOW,0,1,35,351
